# iCatcher annotations --> looking time pipeline

This script, and the helper classes that it calls, are based on Raz et al., 2024 (https://osf.io/ndkt6/), and organizes iCatcher annotations and other subject and trial level data into a format that can feed into our R looking time pipeline. It does this by organizing subject, trial, and look level data in the same format as it is usually outputed in by Datavyu manual coding. 

## Inputs: 

### 1. Lookit study-level JSON file: 

We parse the Lookit study-wide .json log (that is stored at /data/metadata/lookit_study.json), to obtain trial onsets/offsets, trial data, subject data and session data, all in one go.
    
### 2. iCatcher annotation file/s (.npz file per subject, per session)

These are the main outputs from running iCatcher. Expects one file per trial per subject, per session, named accordingly. 

### 3. raw video files (or video-parsed .json files)

needed to extract frame rates for conversion from frame rates to ms (to get look events onset/offset relative to start of video)
    
if this conversion has already been run, .json files with the relevant information should exist in the videos directory. if not, these .json files will be written in the process of obtaining this info 
   
Since we have a separate video for each trial we are not worried about calculating the exact trial onset time. However, we do store the lag between the onset of the audio and the video recording on each trial to use in our main looking time analysis in R. The original Raz et al., 2024 code (https://osf.io/ndkt6/) provides detailed instructions on how to deal with trial onsets relative to the start of videos, both when a manual onset CSV file is required and is not. The original code also provides flexibility with subject level and experiment onset information.
    

In [1]:
%load_ext autoreload
%autoreload 2

## import libraries 

In [2]:
import os
import os.path as op
from pathlib import Path
import sys
import pandas as pd
import numpy as np

module_path = os.path.abspath(os.path.join('../..'))
if module_path not in sys.path:
    sys.path.append(module_path)

from helperfuncs.video_framerates import get_frame_information
from helperfuncs.lookit_json_parser import get_lookit_trial_times

from config import *

## Set relevant paths

In [3]:
raw_data_dir = op.join(SERVER_PATH, 'data', 'raw')
metadata_dir = op.join(PROJECT_PATH, 'data', 'metadata')

# sample1/sample2 live together in data/main/; pilot data lives in data/pilot/.
DATA_FOLDER = 'main' if PROJECT_VERSION in VALID_SECTIONS + ['main'] else PROJECT_VERSION

# Always a list; 'main' expands to both samples, anything else is just itself.
SECTIONS_TO_PROCESS = VALID_SECTIONS if PROJECT_VERSION == 'main' else [PROJECT_VERSION]

data_dir = op.join(PROJECT_PATH, 'data', DATA_FOLDER)

# where the raw iCatcher outputs are stored
icatcher_outputs_dir = op.join(raw_data_dir, 'icatcher_annotations')
# where the raw videos are stored
videos_dir = op.join(raw_data_dir, 'original_videos', 'mp4')

# One lookit JSON path per section being processed
lookit_jsons = {s: op.join(PROJECT_PATH, 'data', 'raw', 'lookit', s, 'lookit_study.json')
                for s in SECTIONS_TO_PROCESS}

# combined trial timing info lives in data/main/data_to_analyze/ (or data/pilot/data_to_analyze/)
lookit_trial_info_csv = op.join(data_dir, 'data_to_analyze', 'level-trials_source-lookit_data.csv')
if os.path.exists(lookit_trial_info_csv):
    os.remove(lookit_trial_info_csv)


### Create all necessary functions

#### get all non-hidden files in dir (helper function):

In [4]:
# list all files except those beginning with '.' i.e., hidden files 
def listdir_nohidden(path):
    for root, dirs, files in os.walk(path):  # Walk through each directory and subdirectory
        for f in files:
            if not f.startswith('.') and f.endswith('.npz'):  # Check conditions
                yield os.path.relpath(os.path.join(root, f), start=path) 

## Set parameters for manual error exclusions


In [5]:
include_manual_edits = 0

if include_manual_edits:
    manually_coded_sections = pd.read_csv("manual_coding_timestamps.csv")

#### convert iCatcher annotated look-events from frame-wise to timing (ms from video onset) 

In [6]:
def subsample_data(data, target_length):
    """
    Subsample the input data to match the target length.
    
    Parameters:
        data (array-like): The input data to subsample.
        target_length (int): The desired length of the output data.
    
    Returns:
        array-like: Subsampled data.
    """
    factor = len(data) / target_length
    indices = np.round(np.arange(0, len(data), factor)).astype(int)[:target_length]
    return data[indices]

def read_convert_output(filename, stamps,manual_edits_df=None):
    """
    Given an npz file containing icatcher annotated frames and looks,
    converts to pandas DataFrame with another column mapping each frame
    to its time stamp in the video
    
    INPUTS: 
    filename (string): name of tabulated iCatcher output file in format
    '[CHILD_ID]/[TRIAL_ID]_[CHILD_ID].npz'
    stamps (List[int]): time stamp for each frame, where stamps[i] is the 
    time stamp at frame i (determined in function get_frame_information(), IMPORTED function)
    
    OUTPUTS: 
    rtype: DataFrame
    
    """
    npz = np.load(filename)
    df = pd.DataFrame([])
    lst = npz.files
    print(lst)
    npz_data = npz[lst[0]]
    print(npz_data)
    confidence_data = npz[lst[1]]

    # Align frame rates of iCatcher and mp4 videos, frame rates coming in are variable because of mp4 conversion
    if len(npz_data) != len(stamps):
        if len(npz_data) > len(stamps):
            npz_data = subsample_data(npz_data, len(stamps))
            confidence_data = subsample_data(confidence_data, len(stamps))
        else:
            stamps = subsample_data(stamps, len(npz_data))
    
    df['frame'] = range(1, len(npz_data) + 1)
    df['lookType'] = npz_data
    # {'noface': -2, 'nobabyface': -1, 'away': 0, 'left': 1, 'right': 2}
    df['lookType'] = df['lookType'].clip(lower=0)
    # 'left' is coded as 1, 'right' as 2, we need to switch those since the video coming in from iCatcher+ is mirrored
    df['lookType'] = df['lookType'].replace({1: 'right', 2: 'left', 0: 'away'})
    
    # convert frames to ms using frame rate
    df['time_ms'] = stamps
    df['time_ms'] = df['time_ms'].astype(int)
    
    df['confidence'] = confidence_data
    
    # SET left/right/away error FRAMES based on manual indexing, from CSV  
    if include_manual_edits: 
        if len(manual_edits_df):
            for index, row in manual_edits_df.iterrows():
                onset = row['onset']
                offset = row['offset']
                df.loc[onset:offset, 'lookType'] = row['value']
                df.loc[onset:offset, 'confidence'] = -1

    # split into dfs based on when the change happens
    df['group'] = df['lookType'].ne(df['lookType'].shift()).cumsum()
    df_grps = df.groupby('group')
    
    dfs = []
    for _, data in df_grps:
        dfs.append(data)


    looks_onoff_grouped = pd.DataFrame()
    
    for grpI in range(len(dfs)):
        indices = dfs[grpI].index.tolist()
        # first row time_ms is onset, row after last row time_ms is offset 
        onset = df.iloc[indices[0]].time_ms

        if grpI == len(dfs)-1:
            # assume dur of last frame is the mean dur of frames 
            dur = np.floor(np.mean(np.diff(df['time_ms'])))
            offset = onset + dur
        else:
            offset = df.iloc[indices[-1]+1].time_ms
        
        lookType = df.iloc[indices[0]].lookType
        confidence = np.mean(df.iloc[indices].confidence)
    
        cur_lookinfo = pd.DataFrame({"Looks.ordinal": grpI,
                    "Looks.onset" : onset,
                    "Looks.offset": offset, 
                    "Looks.lookType": lookType,
                    "Looks.confidence": confidence},  index=[0])

        looks_onoff_grouped = pd.concat([looks_onoff_grouped, cur_lookinfo], ignore_index=True)


    
    return [looks_onoff_grouped, df]

In [7]:
def get_lookit_info(child_id, trial_id, session_id, trial_info_file):

    if Path(lookit_trial_info_csv).is_file():
        # Combined file exists — read and filter to current child/trial
        df = pd.read_csv(trial_info_file)

    else:
        # Parse each section's JSON, tag with section name, and write the combined file
        os.makedirs(os.path.dirname(lookit_trial_info_csv), exist_ok=True)
        dfs = []
        for section, json_path in lookit_jsons.items():
            section_df = get_lookit_trial_times(json_path, section)
            section_df['section'] = section
            dfs.append(section_df)
        df_out = pd.concat(dfs, ignore_index=True)

        if op.exists(lookit_trial_info_csv):
            existing = pd.read_csv(lookit_trial_info_csv)
            # Drop rows for sections being re-processed so we don't duplicate
            if 'section' in existing.columns:
                existing = existing[~existing['section'].isin(SECTIONS_TO_PROCESS)]
            df_out = pd.concat([existing, df_out], ignore_index=True)

        df_out.to_csv(lookit_trial_info_csv, index=False)
        df = df_out

    # get part of df from current child
    df = df[df['SubjectInfo.subjID'] == child_id]

    # get part of df from current trial
    df = df[df['Trials.trialID'] == trial_id]

    if 'SubjectInfo.sessionNumber' in df:
        df = df[df['SubjectInfo.sessionNumber'] == session_id]
    else:
        df['SubjectInfo.sessionNumber'] = 1

    df = df.reset_index(drop=True)

    return df


In [8]:
for section, json_path in lookit_jsons.items():
    print(f"--- {section} ---")
    get_lookit_trial_times(lookit_json=json_path, proj_version=section)


--- sample1 ---
49-great-job
2-study-intro
3-video-config
4-video-consent
8-exp-get-ready
7-webcam-display
12-easy-snail-cow
21-easy-acorn-key
9-exp-calibration
10-easy-cheese-mud
16-hard-potato-pot
41-hard-snail-worm
11-easy-turkey-swan
20-hard-turkey-goat
23-hard-turtle-frog
0-eligibility-survey
38-easy-turtle-horse
5-study-instructions
6-setup-instructions
35-hard-acorn-coconut
47-hard-cheese-butter
50-parent-exit-survey
28-easy-potato-glasses
42-easy-squirrel-eagle
1-eligibility-procedure
32-hard-squirrel-monkey
27-easy-bulldozer-orange
1-eligibility-procedure-0
31-hard-bulldozer-tractor
14-attention-getter-meadow
24-attention-getter-balloons
37-easy-snail-cow-distractor
39-attention-getter-balloons
40-easy-acorn-key-distractor
26-hard-snail-worm-distractor
29-attention-getter-mountains
30-easy-cheese-mud-distractor
33-hard-potato-pot-distractor
44-attention-getter-mountains
36-easy-turkey-swan-distractor
43-hard-turkey-goat-distractor
45-hard-turtle-frog-distractor
13-easy-turtle-

#### get trial onsets w/r/t video

In [22]:
def get_trial_sets(child_id, trial_id, session_id, trial_info_file):
    """
    Finds corresponding trial info 
    and returns a list of [onset, offset] times for each trial in 
    milliseconds, with respect to video onset

    """

    lookit_df = get_lookit_info(child_id, trial_id, session_id, trial_info_file)
    df = lookit_df[[col for col in lookit_df.columns if col.startswith('Trials.') or col == "section"]]    
    df = df.copy()
    df['Trials.ordinal'] = df.index.values.tolist()
    
    return df

#### Get subject level data 

In [23]:
def get_subject_info(child_id, trial_id, session_id, trial_info_file): 
        
    lookit_df = get_lookit_info(child_id, trial_id, session_id, trial_info_file)
    df = lookit_df[[col for col in lookit_df.columns if col.startswith('SubjectInfo.')  or col == "section"]]       
    if 'SubjectInfo.sessionNumber' in df:
        df = df[df['SubjectInfo.sessionNumber'] == session_id] 
    else:
        df['SubjectInfo.sessionNumber'] = 1
        
    return df

In [11]:
print(SERVER_PATH)

/Volumes/vislearnlab/experiments/visual-precision


### main function: run processes 

In [12]:
# set up output — combined file lives in data/main/data_to_analyze/ (or data/pilot/...)
output_data_dir = op.join(PROJECT_PATH, 'data', DATA_FOLDER, 'data_to_analyze')
fname_final_output = op.join(output_data_dir, 'level-looks_source-icatcher_data.csv')

# Build the list of already-processed (child, trial) pairs.
# When running a single section, filter to that section so re-runs only skip that section's rows.
# When running 'main' (multiple sections), consider all existing rows.
if op.exists(fname_final_output):
    existing_df = pd.read_csv(fname_final_output)
    if len(SECTIONS_TO_PROCESS) == 1 and 'section' in existing_df.columns:
        section_df = existing_df[existing_df['section'] == SECTIONS_TO_PROCESS[0]]
    else:
        section_df = existing_df
    processed_participants = list(zip(section_df['SubjectInfo.subjID'], section_df['Trials.trialID']))
else:
    processed_participants = []

if not any(listdir_nohidden(icatcher_outputs_dir)):
    print("No iCatcher files found, make sure you are connected to your server")
for filename in listdir_nohidden(icatcher_outputs_dir):
    # Only include files that are placed in folders set up in a directory for a particular child
    if '/' in filename and ('easy' in filename or 'hard' in filename):
        # Split into child_id and trial id
        child_id, trial_id_with_extension_child_id = filename.split('/', 1)
        # Remove the .npz extension from trial id
        trial_id_with_child_id = trial_id_with_extension_child_id.rsplit('.', 1)[0]
        trial_id = trial_id_with_child_id.split("_")[0]
    else:
        continue
    print('child: ', child_id)
    print('trial: ', trial_id)
    # Need to pull session information in the future, our current experiment only uses a single session.
    session_id = 1
    # Check if this subject's data already exists in output file
    if (child_id, trial_id) in processed_participants:
        print(f'Skipping {child_id}, {trial_id} - already exists in output file')
        continue
    # determine trial info files
    trial_info_file = lookit_trial_info_csv

    # get trial onsets and offsets from input file, match to iCatcher file
    trials_df = get_trial_sets(child_id, trial_id, session_id, trial_info_file)
    trials_df = trials_df.reset_index(drop=True)
    if (trials_df.empty):
        print(f'Skipping {child_id}, {trial_id} - no trial info found')
        continue

    if include_manual_edits:
        manual_edits_df = manually_coded_sections[manually_coded_sections['child_id'] == child_id and manually_coded_sections['trial_id'] == trial_id]

    # determine video source
    vid_path = op.join(videos_dir, child_id, f"{trial_id}_{child_id}.mp4")
    json_video_data = op.join(videos_dir, child_id, f"{trial_id}_{child_id}.json")
    # get timestamp for each frame in the video
    timestamps, length = get_frame_information(vid_path, json_video_data)
    if not timestamps:
        print('video not found for {} in {} folder'.format(child_id, videos_dir))
        continue

    # initialize df with time stamps for iCatcher file
    icatcher_path = icatcher_outputs_dir + '/' + filename

    if include_manual_edits:
        [looks_df_grouped, looks_df] = read_convert_output(icatcher_path, timestamps, manual_edits_df)
    else:
        [looks_df_grouped, looks_df] = read_convert_output(icatcher_path, timestamps)

    looks_df_grouped = looks_df_grouped[['Looks.ordinal', 'Looks.onset', 'Looks.offset', 'Looks.lookType', 'Looks.confidence']]
    looks_df_grouped = looks_df_grouped.reset_index(drop=True)

    # make subject level dataframe
    subject_info = get_subject_info(child_id, trial_id, session_id, trial_info_file)
    subject_info = subject_info.loc[[0], :].copy()
    subject_info['SubjectInfo.subjID'] = subject_info['SubjectInfo.subjID'].astype(str)
    subject_info = subject_info.reset_index(drop=True)

    df_concat = pd.concat([pd.DataFrame(np.repeat(subject_info.values, len(looks_df_grouped), axis=0), columns=subject_info.columns),
                           looks_df_grouped, pd.DataFrame(np.repeat(trials_df.values, len(looks_df_grouped), axis=0), columns=trials_df.columns)], axis=1)

    # add trial_id, child_id, and section to the dataframe
    looks_df['Trials.trialID'] = trial_id
    looks_df['SubjectInfo.subjID'] = child_id

    # Section is always tagged: infer from the trial info CSV (set during JSON parsing)
    if 'section' in trials_df.columns and not trials_df['section'].empty:
        looks_df['section'] = trials_df['section'].iloc[0]

    if not os.path.exists(output_data_dir):
        os.makedirs(output_data_dir)
    # Append or create file
    if os.path.exists(fname_final_output):
        looks_df.to_csv(fname_final_output, mode='a', header=False, index=False)  # Append without header
    else:
        looks_df.to_csv(fname_final_output, index=False)  # Create new file


child:  VVI078
trial:  hard-acorn-coconut-distractor
Skipping VVI078, hard-acorn-coconut-distractor - already exists in output file
child:  VVI078
trial:  hard-snail-worm-distractor
Skipping VVI078, hard-snail-worm-distractor - already exists in output file
child:  VVI078
trial:  easy-squirrel-eagle
Skipping VVI078, easy-squirrel-eagle - already exists in output file
child:  VVI078
trial:  easy-cheese-mud-distractor
Skipping VVI078, easy-cheese-mud-distractor - already exists in output file
child:  VVI078
trial:  hard-potato-pot
Skipping VVI078, hard-potato-pot - already exists in output file
child:  VVI078
trial:  easy-cheese-mud
Skipping VVI078, easy-cheese-mud - already exists in output file
child:  VVI078
trial:  easy-potato-glasses
Skipping VVI078, easy-potato-glasses - already exists in output file
child:  VVI078
trial:  hard-turkey-goat-distractor
Skipping VVI078, hard-turkey-goat-distractor - already exists in output file
child:  VVI078
trial:  hard-squirrel-monkey-distractor
S

In [ ]:
import pandas as pd

df = pd.read_csv(fname_final_output)
cols = df.columns.tolist()
# ['section', 'frame', 'lookType', 'time_ms', 'confidence', 'group', 'Trials.trialID', 'SubjectInfo.subjID']

records = df.to_dict('records')

fixed_records = []
for rec in records:
    vals = list(rec.values())
    if pd.isna(vals[-1]):  # SubjectInfo.subjID is NaN -> missing 'section' col
        # Current:  [frame, lookType, time_ms, confidence, group, trialID, subjID, NaN]
        # Correct:  [section='sample2', frame, lookType, time_ms, confidence, group, trialID, subjID]
        vals = ['sample2'] + vals[:-1]  # drop trailing NaN, prepend section
        rec = dict(zip(cols, vals))
    fixed_records.append(rec)

df_fixed = pd.DataFrame(fixed_records)

# Verify
assert df_fixed['SubjectInfo.subjID'].isna().sum() == 0, "Still have misaligned rows!"
print(df_fixed[df_fixed['SubjectInfo.subjID'] == 'VVI177'].head(10))


/var/folders/xc/86qyqqhn04zg02by5wt4_wdr0000gn/T/ipykernel_38923/556329582.py:3: DtypeWarning: Columns (0: frame, 1: group) have mixed types. Specify dtype option on import or set low_memory=False.
  df = pd.read_csv(fname_final_output)


        section frame lookType time_ms  confidence group  \
975336  sample2     1     away       0   -1.000000   1.0   
975337  sample2     2     away      48   -1.000000   1.0   
975338  sample2     3     away      80   -1.000000   1.0   
975339  sample2     4     away     112   -1.000000   1.0   
975340  sample2     5     away     144    0.627420   1.0   
975341  sample2     6     away     178    0.942049   1.0   
975342  sample2     7     away     208    0.928618   1.0   
975343  sample2     8     away     240    0.964037   1.0   
975344  sample2     9     away     272    0.743871   1.0   
975345  sample2    10     away     304    0.837188   1.0   

                      Trials.trialID SubjectInfo.subjID  
975336  easy-bathtub-sock-distractor             VVI177  
975337  easy-bathtub-sock-distractor             VVI177  
975338  easy-bathtub-sock-distractor             VVI177  
975339  easy-bathtub-sock-distractor             VVI177  
975340  easy-bathtub-sock-distractor             

In [26]:
df_fixed.to_csv(fname_final_output, index=False)